# Proyecto Oráculo · Notebook de evaluación · Grupo G01

**Análisis y Diseño de Algoritmos · Universidad de Caldas · 2026-2** — Integrante: Juan Manuel Carmona Villegas

Evalúa la configuración de [`entrega.json`](https://github.com/Manuel29296/ADA-2026-2-GP01/blob/main/entrega.json) sobre un conjunto de prueba, **sin editar código, rutas ni parámetros**.

**Cómo usarlo**

1. *Entorno de ejecución ▸ Cambiar tipo de entorno de ejecución ▸* **T4 GPU**.
2. *Entorno de ejecución ▸* **Ejecutar todo**.
3. La primera vez, la celda 1 instala dependencias y **reinicia el entorno sola**. Cuando reconecte, use otra vez **Ejecutar todo**.
4. Datos de prueba: si hay un archivo `.json` o `.jsonl` en `/content` (por ejemplo `datos_prueba.json`), se usa ese. Si no, la celda 4 abre un botón para subirlo. Si no se sube nada, se ejecuta en **modo demostración** con las 87 instancias de validación del curso.

Formato esperado de cada instancia: el mismo de `datos_visibles.json` (`prompt`, `ids`, `kwargs`; opcionales `id`, `restriccion`). Se acepta una lista JSON, un objeto con una lista dentro o JSONL.

## 1 · Instalación (solo la primera vez; reinicia el entorno)

In [ ]:
import os
os.chdir("/content")
MARCA = "/content/.g01_instalado"

if not os.path.exists(MARCA):
    !pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate nltk spacy emoji langdetect immutabledict
    !python -m spacy download en_core_web_sm -q
    # Datos de nltk que usan los verificadores (su descargador falla con el proxy de Colab)
    import io, urllib.request, zipfile
    NLTK = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"
    for carpeta, nombre in [("tokenizers", "punkt"), ("tokenizers", "punkt_tab"),
                            ("taggers", "averaged_perceptron_tagger"),
                            ("taggers", "averaged_perceptron_tagger_eng")]:
        with urllib.request.urlopen(f"{NLTK}/{carpeta}/{nombre}.zip") as r:
            zipfile.ZipFile(io.BytesIO(r.read())).extractall(f"/root/nltk_data/{carpeta}")
    open(MARCA, "w").close()
    print("Instalación lista. El entorno se reinicia ahora: cuando reconecte, use de nuevo 'Ejecutar todo'.")
    os.kill(os.getpid(), 9)
else:
    print("Dependencias ya instaladas.")

## 2 · Archivos del repositorio del grupo y verificadores

In [ ]:
import os
os.chdir("/content")
REPO = "https://raw.githubusercontent.com/Manuel29296/ADA-2026-2-GP01/main"

# Siempre la versión publicada en GitHub (no copias locales viejas)
for archivo in ["oraculo.py", "ayudas.py", "utilidades.py", "entrega.json", "datos_visibles.json"]:
    !wget -q --no-cache -O {archivo} {REPO}/{archivo}

# Verificadores de restricciones: IFEvalG (el que usa oraculo.py) e IFBench (familias adicionales)
if not os.path.exists("/content/open-instruct"):
    !git clone -q --depth 1 https://github.com/allenai/open-instruct /content/open-instruct
if not os.path.exists("/content/IFBench"):
    !git clone -q --depth 1 https://github.com/allenai/IFBench /content/IFBench
print("Archivos listos:", sorted(f for f in os.listdir() if f.endswith((".py", ".json"))))

## 3 · La configuración entregada

In [ ]:
from utilidades import cargar_entrega, RANURAS

GRUPO, SEMANA, CONFIG = cargar_entrega("entrega.json")
print(f"Grupo {GRUPO} · semana {SEMANA}")
print("Configuración:", CONFIG)

## 4 · Los datos de prueba

In [ ]:
import glob, os
from utilidades import leer_instancias

PREFERIDOS = ["datos_prueba", "datos_privados", "test_privado", "test", "prueba"]
PROPIOS = {"datos_visibles.json", "entrega.json"}

candidatos = [f"/content/{n}.{e}" for n in PREFERIDOS for e in ("json", "jsonl")]
candidatos += sorted(p for p in glob.glob("/content/*.json") + glob.glob("/content/*.jsonl")
                     if os.path.basename(p) not in PROPIOS and "cache" not in os.path.basename(p) and not os.path.basename(p).startswith("resultados"))
RUTA = next((p for p in candidatos if os.path.exists(p)), None)

if RUTA is None:
    try:
        from google.colab import files
        print("Suba el archivo de datos de prueba (JSON o JSONL). Si cancela, se usa el modo demostración.")
        subidos = files.upload()
        if subidos:
            RUTA = "/content/" + next(iter(subidos))
    except Exception as e:
        print("No se pudo abrir el selector de archivos:", e)

if RUTA:
    MODO = "prueba"
    INSTANCIAS = leer_instancias(RUTA)
else:
    MODO = "demostración (validación del curso)"
    INSTANCIAS = [x for x in leer_instancias("datos_visibles.json") if x.get("split") == "validacion"]

print(f"Modo: {MODO} · archivo: {RUTA or 'datos_visibles.json'} · {len(INSTANCIAS)} instancias")

## 5 · Verificadores y modelo

Antes de generar se revisa que cada restricción tenga verificador. Las instancias sin verificador se informan y cuentan como fallo en la precisión sobre el archivo completo; así la evaluación no se detiene a mitad de camino.

In [ ]:
import sys
from collections import Counter
from oraculo import Oraculo, _REG
from ayudas import cargar_modelo

EXTRA = []
try:
    sys.path.insert(0, "/content/IFBench")
    import instructions_registry as IFBENCH
    EXTRA.append(IFBENCH.INSTRUCTION_DICT)
    print("IFBench: cargado")
except Exception as e:
    print("IFBench no disponible; se usa solo IFEvalG:", type(e).__name__, e)

CONOCIDAS = set(_REG.INSTRUCTION_DICT).union(*[set(r) for r in EXTRA])
CALIFICABLES = [x for x in INSTANCIAS if all(i in CONOCIDAS for i in x["ids"])]
SIN_VERIFICADOR = Counter(i for x in INSTANCIAS for i in x["ids"] if i not in CONOCIDAS)
print(f"Instancias calificables: {len(CALIFICABLES)} de {len(INSTANCIAS)}")
if SIN_VERIFICADOR:
    print("Familias sin verificador:", dict(SIN_VERIFICADOR))

modelo = cargar_modelo("qwen17b")      # el mismo modelo con el que se buscó la configuración
oraculo = Oraculo(modelo, CALIFICABLES, cache="/content/cache_evaluacion_G01.json",
                  lote=16, presupuesto=32_000_000, registros=EXTRA)

## 6 · Evaluación

In [ ]:
import time, json, csv

t0 = time.perf_counter()
R = oraculo.evaluar(CONFIG, CALIFICABLES, semilla=1)
SEGUNDOS = time.perf_counter() - t0

aciertos = round(R.precision * len(CALIFICABLES))
precision_archivo = aciertos / len(INSTANCIAS)
fallos = Counter(t["violo"] for t in R.trazas)

print("=" * 62)
print(f" Grupo {GRUPO} · modo {MODO}")
print(f" Configuración: {CONFIG}")
print(f" Precisión (instancias calificables): {R.precision:.1%}  ({aciertos}/{len(CALIFICABLES)})")
print(f" Precisión (archivo completo):        {precision_archivo:.1%}  ({aciertos}/{len(INSTANCIAS)})")
print(f" Tiempo: {SEGUNDOS/60:.1f} min")
print("=" * 62)
print("Restricciones que más fallaron (primera violada por instancia):")
for familia, veces in fallos.most_common(10):
    print(f"  {veces:4d}  {familia}")

resultado = {"grupo": GRUPO, "semana": SEMANA, "config": CONFIG, "modo": MODO,
             "archivo": RUTA or "datos_visibles.json (validación)",
             "instancias": len(INSTANCIAS), "calificables": len(CALIFICABLES),
             "aciertos": aciertos, "precision_calificables": R.precision,
             "precision_archivo": precision_archivo, "segundos": round(SEGUNDOS, 1),
             "fallos_por_familia": dict(fallos), "familias_sin_verificador": dict(SIN_VERIFICADOR)}
json.dump(resultado, open("resultados_evaluacion_G01.json", "w"), indent=1, ensure_ascii=False)

violo = {t["id"]: t["violo"] for t in R.trazas}
with open("resultados_evaluacion_G01.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id", "acierto", "primera_restriccion_violada"])
    for x in CALIFICABLES:
        w.writerow([x["id"], int(x["id"] not in violo), violo.get(x["id"], "")])
print("\nGuardado: resultados_evaluacion_G01.json y resultados_evaluacion_G01.csv")

## 7 · Ejemplos de fallos y descarga de resultados

In [ ]:
for t in R.trazas[:3]:
    print("violó:", t["violo"])
    print(t["salida"][:300])
    print("-" * 60)

try:
    from google.colab import files
    files.download("resultados_evaluacion_G01.json")
    files.download("resultados_evaluacion_G01.csv")
except Exception:
    pass